In [ ]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

In [ ]:
print(
    os.getenv("NEO4J_URI"),
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
    os.getenv("NEO4J_DATABASE")
)

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph()

# 의미기반 검색

## 노드에 임베딩 필드 추가

### 임베딩 필드의 필요성

Neo4j는 기본적으로 **키워드 기반 검색**을 지원한다. 예를 들어 영화 제목에 "인셉션"이라는 단어가 포함된 노드를 찾는 것은 가능하지만, "꿈속에서 벌어지는 심리 스릴러"와 같은 **의미 기반 질의**로는 원하는 노드를 찾을 수 없다.

이 한계를 극복하기 위해 노드에 **임베딩 필드**(**Embedding Field**)를 추가한다.

- 노드의 텍스트 데이터(제목, 줄거리, 설명 등)를 **고차원 벡터**로 변환하여 속성으로 저장한다.
- 벡터는 텍스트의 **의미(Semantic)**를 수치로 표현한 것으로, 의미가 유사한 텍스트일수록 벡터 간 거리가 가까워진다.
- 임베딩 필드에 **Vector Index**를 생성하면, 사용자의 질의와 **의미적으로 유사한 노드**를 효율적으로 검색할 수 있다.

> **예시**
> "우주를 배경으로 한 모험 영화"라는 질의를 벡터로 변환하면,
> 줄거리에 해당 내용이 담긴 Movie 노드와 높은 유사도를 갖게 되어 검색 결과로 반환된다.

### 임베딩 필드를 만드는 데 필요한 것

임베딩 필드를 노드에 추가하려면 아래 세 가지가 필요하다.

- **임베딩 대상 텍스트**
의미기반 검색(vector검색)에 사용될 내용의 텍스트를 준비한다. 기존 텍스트 속성들의 값을 이용할 수도 있고 새로운 내용일 수도 있다.

- **임베딩 모델**

선택한 텍스트를 고차원 벡터로 변환해 줄 **외부 임베딩 모델**이 필요하다.
- Neo4j는 벡터를 직접 생성하지 않으므로, OpenAI 등의 임베딩 API를 별도로 호출해야 한다.
- 사용하는 모델에 따라 출력 벡터의 **차원 수**가 결정된다.

- **벡터를 저장할 노드 속성**

변환된 벡터값을 노드에 **새로운 속성**으로 저장한다.
- 예) `m.content_embedding = [0.12, -0.45, ...]`
- Vector를 저장하는 속성은 **벡터 인덱스로 설정**해야 유사도 검색에 활용할 수 있다.

## 벡터 인덱스 생성

### 벡터 인덱스의 역할

벡터 인덱스는 고차원 벡터 공간에서 **유사한 벡터를 빠르게 검색**하기 위한 특수 인덱스이다.

- 임베딩 벡터는 수백~수천 개의 숫자로 구성된 고차원 데이터이다. 인덱스 없이 유사도를 계산하려면 **모든 노드를 순차적으로 비교**해야 하므로 매우 비효율적이다.
- 벡터 인덱스를 사용하면 **ANN(Approximate Nearest Neighbor)** 알고리즘을 통해 유사도가 높은 벡터를 빠르게 탐색할 수 있다.
- 벡터 인덱스를 통해 **의미적으로 유사한 노드**를 효율적으로 찾을 수 있다.
  - 예) "액션 영화 추천" 쿼리 벡터와 가장 유사한 Movie 노드를 검색

### 벡터 인덱스 생성 구문

```cypher
CREATE VECTOR INDEX <인덱스명>
FOR (n:<레이블>) ON (n.<속성명>)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: <차원수>,
    `vector.similarity_function`: '<유사도함수>'
  }
}
```

**구문 설명**

| 항목 | 설명 |
|------|------|
| `<인덱스명>` | 인덱스를 식별하는 이름. 이후 검색 쿼리에서 참조함 |
| `FOR (n:<레이블>)` | 인덱스를 적용할 노드 레이블 지정 |
| `ON (n.<속성명>)` | 임베딩 벡터가 저장된 속성명 지정 |
| `vector.dimensions` | 임베딩 모델의 출력 차원 수. 저장된 벡터 차원과 반드시 일치해야 함 |
| `vector.similarity_function` | 벡터 간 유사도 계산 방식 지정 |

**`vector.similarity_function` 옵션**

| 값 | 설명 | 특징 |
|----|------|------|
| `cosine` | 코사인 유사도 | 벡터의 **방향**을 기준으로 유사도 측정. 텍스트 임베딩에 가장 일반적으로 사용 |
| `euclidean` | 유클리드 거리 | 벡터 간 **거리**를 기준으로 유사도 측정 |

### 예시

Movie 노드의 `embedding` 속성에 대해 1536차원, 코사인 유사도 기반 벡터 인덱스를 생성한다.

```cypher
CREATE VECTOR INDEX movie_embedding_index
FOR (m:Movie) ON (m.content_embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
}
```

> **차원 수와 임베딩 모델의 관계**
> 차원 수는 사용하는 임베딩 모델에 따라 결정된다.
> - `text-embedding-3-small` → 1536차원
> - `text-embedding-3-large` → 3072차원
>
> 인덱스의 차원 수와 실제 저장된 벡터의 차원 수가 일치하지 않으면 오류가 발생한다.


### 인덱스 확인

생성된 벡터 인덱스 목록은 아래 명령으로 확인한다.

```cypher
SHOW VECTOR INDEXES
```

### 노드의 임베딩 속성에 임베딩 벡터 저장
- `db.create.setNodeVectorProperty()` 함수(프로시저) 를 사용한다.

    ```cypher
        CALL db.create.setNodeVectorProperty(node, propertyName, vector)
    ```

#### 파라미터

| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `node` | Node | 벡터를 저장할 대상 노드 |
| `propertyName` | String | 저장할 속성명 |
| `vector` | List\<Float\> | 저장할 임베딩 벡터값 |


#### 사용 예시

```cypher
MATCH (m:Movie {title: "Inception"})
CALL db.create.setNodeVectorProperty(m, 'embedding', [0.12, -0.34, 0.56, ...])
```

In [ ]:
##########################################################################################
# 벡터 인덱스 생성 - content_embedding 속성에 대한 Vector Index 생성
##########################################################################################
create_vector_index_query = """
CREATE VECTOR INDEX movie_content_embeddings IF NOT EXISTS 
FOR (m:Movie) ON m.content_embedding 
OPTIONS {
  indexConfig: {
    `vector.dimensions`: $size,
    `vector.similarity_function`: 'cosine'
  }
}
"""
graph.query(create_vector_index_query, params={"size":1536})

In [ ]:
# 벡터 인덱스 확인
check_vector_index_query = """
SHOW VECTOR INDEXES
"""
vector_indexes = graph.query(check_vector_index_query)
for index in vector_indexes:
    # 벡터 인덱스 정보 출력
    print(f"Index Name: {index['name']}")
    print(f"Type: {index['type']}")    
    print(f"Property Key: {index['properties']}")
    print("-" * 40)

In [ ]:
#######################################################################################
# content_embedding 에 영화 제목과 줄거리, 태그라인을 결합하여 임베딩 생성 후 저장
#######################################################################################
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

movies_query = """
MATCH (m:Movie)
WHERE m.title IS NOT NULL
RETURN m.id AS id, m.title AS title, m.overview AS overview, m.tagline AS tagline
"""
movies = graph.query(movies_query)

BATCH_SIZE = 100

##################################################
#  임베딩 생성 및 저장 (배치 처리)
##################################################
for i in range(0, len(movies), BATCH_SIZE):
    batch = movies[i:i+BATCH_SIZE]
    batch_texts = []
    batch_ids = []
    
    # 배치 데이터 준비
    for movie in batch:
        # title, overview, tagline을 "\n\n"으로 결합.
        content_text = f"{movie['title']}"
        if movie['tagline']:
            content_text += f"\n\n{movie['tagline']}"
        if movie['overview']:
            content_text += f"\n\n{movie['overview']}"
        
        if content_text.strip():
            batch_texts.append(content_text)
            batch_ids.append(movie['id'])
    
    try:
        if batch_texts:
            batch_embeddings = embeddings.embed_documents(batch_texts)
            
            # UNWIND를 사용한 배치 업데이트
            batch_data = [{"id": article_id, "embedding": embedding_vector} 
                         for article_id, embedding_vector in zip(batch_ids, batch_embeddings)]
            
            batch_update_query = """
            // UNWIND를 사용하여 배치 데이터를 개별 행으로 변환
            UNWIND $batch AS item

            // 영화 ID로 해당 Movie 노드 찾기
            MATCH (m:Movie {id: item.id})

            // db.create.setNodeVectorProperty 프로시저를 호출하여 벡터 속성 설정
            // 첫 번째 인자: 대상 노드, 두 번째 인자: 속성 이름, 세 번째 인자: 벡터 값
            CALL db.create.setNodeVectorProperty(m, 'content_embedding', item.embedding)

            // 업데이트된 노드 수 반환
            RETURN count(m) as updated
            """
            
            result = graph.query(batch_update_query, params={"batch": batch_data})
            print(f"배치 처리 완료: {i+1}~{min(i+len(batch_texts), len(movies))} / {len(movies)}, 업데이트됨: {result[0]['updated']}")
            
    except Exception as e:
        print(f"배치 임베딩 생성 실패 (배치 인덱스 {i}): {str(e)}")

print(f"영화 임베딩 업데이트 완료!! 총 {len(movies)}개 처리")

## 의미 기반 검색(Semantic Search)
- **SEARCH** 절을 사용한다.
  - SEARCH는 독립된 절이 아니라, MATCH/OPTIONAL MATCH에 종속된 서브절이다.
  ```cypher
    [OPTIONAL] MATCH (조회할 노드)
    SEARCH 노드 IN (
        VECTOR INDEX vector_index_name
        FOR embedding_vector      // query vector
        [WHERE 필터조건]           // in-index filtering-속성조회조건 (옵션)
        LIMIT 조회개수             // 필수: top_k, INTEGER
    )
    [SCORE AS 별칭]                // 유사도 점수 반환 (옵션)
  ```
  - `[ ]` 옵션
  **예**
  ```cypher
    MATCH (m:Movie)
    SEARCH m IN (
        VECTOR INDEX movie_content_embeddings
        FOR $query_embedding
        WHERE m.released >= 2000
        LIMIT 5
    )
    SCORE AS similarity
    RETURN m.title, similarity
  ```

- 구버전 방식 (deprecated)
  - `db.index.vector.queryNodes()` 프로시저 사용
    ```cypher
        CALL db.index.vector.queryNodes('movie_content_embeddings', $top_k, $query_embedding)
        YIELD node, score
    ```

In [ ]:
def semantic_movie_search(search_text:str, top_k:int=5)-> list:
    """
    텍스트 쿼리를 받아 의미적으로 가장 유사한 영화를 반환합니다.
    
    매개변수:
        search_text (str): 검색할 텍스트 쿼리
        top_k (int): 반환할 최대 결과 수 (기본값: 5)
        
    반환값:
        list: 유사도 점수가 높은 순으로 정렬된 영화 정보 목록
    """
    # 검색 텍스트의 임베딩 생성 (OpenAI API를 사용하여 텍스트를 벡터로 변환)
    query_embedding = embeddings.embed_query(search_text)
    
    # Neo4j 벡터 검색 쿼리 실행
    vector_search_query = """
    MATCH (m:Movie)

    SEARCH m IN (
        VECTOR INDEX movie_content_embeddings
        FOR $query_embedding
        LIMIT $top_k
    )
    SCORE AS similarity
    
    RETURN m.title AS title,
           m.released AS released,
           m.rating AS rating,
           similarity
    
    ORDER BY similarity DESC;
    """

    results = graph.query(
        vector_search_query,
        params={"top_k": top_k, "query_embedding": query_embedding}
    )
    
    return results

In [ ]:
# 임베딩 기반 의미 검색
query = "a Drama movie about artificial intelligence and reality"
# query = "인공지능과 현실에 관한 SF 영화"
# query = "두 남녀의 아름다운 사랑을 그린 영화"
results = semantic_movie_search(query)
for result in results:

    print(f"{result['title']} - 유사도: {result['similarity']:.4f}, " +
          f"평점: {result['rating']}, 개봉: {result['released']}")